# Notebook 01 — DenPAR Dataset Quality Control

Inspect the DenPAR dataset: count images, visualise sample radiographs,
check annotation coverage, and produce QC statistics.

**Run from the project root:** `opengeotrust-periosam-denpar/`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

DATA_ROOT = '../data/raw/DenPAR'
SPLITS = ['Training', 'Validation', 'Testing']

In [ ]:
# Run dataset inspection
from src.data.inspect_denpar import run_inspect
audit = run_inspect(DATA_ROOT, save_path='../outputs/logs/dataset_audit.json')

In [ ]:
# Summary table
print(f"\n{'Split':<12} {'Images':>8} {'Radio masks':>12} {'Tooth masks':>12} {'Bone JSON':>10} {'KP JSON':>8}")
print('-' * 65)
for split, info in audit.get('splits', {}).items():
    sf = info.get('subfolders', {})
    print(f"{split:<12}"
          f"{sf.get('images', {}).get('count', 0):>8}"
          f"{sf.get('masks_radiograph', {}).get('count', 0):>12}"
          f"{sf.get('masks_tooth', {}).get('count', 0):>12}"
          f"{sf.get('bone_level', {}).get('count', 0):>10}"
          f"{sf.get('keypoints', {}).get('count', 0):>8}")

In [ ]:
# Visualise 6 sample radiographs with masks
from src.data.parse_denpar import load_split

samples = load_split(DATA_ROOT, 'Training', max_samples=6)

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for i, s in enumerate(samples):
    img = np.array(Image.open(s.image_path).convert('L'))
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(s.image_id[:12], fontsize=7)
    axes[0, i].axis('off')
    
    if s.mask_radio_path and s.mask_radio_path.exists():
        mask = np.array(Image.open(s.mask_radio_path).convert('L'))
        axes[1, i].imshow(img, cmap='gray')
        axes[1, i].imshow(mask > 127, alpha=0.4, cmap='Greens')
        axes[1, i].set_title('+ tooth mask', fontsize=7)
    else:
        axes[1, i].imshow(img, cmap='gray')
        axes[1, i].set_title('mask: missing', fontsize=7, color='red')
    axes[1, i].axis('off')

fig.suptitle('DenPAR Training Set — Sample Radiographs + Masks', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/qc_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Image resolution distribution
from PIL import Image as PILImage

widths, heights = [], []
img_dir = Path(DATA_ROOT) / 'Training' / 'Images'
if img_dir.exists():
    for p in list(img_dir.iterdir())[:100]:
        if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}:
            try:
                w, h = PILImage.open(p).size
                widths.append(w); heights.append(h)
            except: pass

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(widths, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Width (px)'); axes[0].set_title('Image Widths')
axes[1].hist(heights, bins=20, color='coral', edgecolor='white')
axes[1].set_xlabel('Height (px)'); axes[1].set_title('Image Heights')
plt.tight_layout()
plt.savefig('../outputs/figures/qc_resolution_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}')
print(f'Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')